# Fixed SARIMA forecasting system

Reproduce the fixed SARIMA forecasting system from raw metadata and readings. Execution takes a few minutes.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

def is_project(path):
    return (path / 'main.py').is_file() and (path / 'src/workload/data.py').is_file()

anchor = Path(os.environ.get('VM_PROJECT_ROOT', str(Path.cwd()))).expanduser().resolve()
ROOT = next((p for p in [anchor, *anchor.parents] if is_project(p)), None)
if ROOT is None:
    matches = [p for p in anchor.iterdir() if p.is_dir() and is_project(p)]
    if len(matches) != 1:
        raise RuntimeError('Open from the checkout/notebooks directory, or set VM_PROJECT_ROOT to the checkout path.')
    ROOT = matches[0]
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'notebooks'))
from workload.data import load_data, split, validate_predictions

# Set VM_DATA_DIR, or change this variable to the directory with the two raw files.
DATA_DIR = Path(os.environ.get('VM_DATA_DIR', str(ROOT / 'challenge_data'))).expanduser()
data = load_data(DATA_DIR)
n, future_times = split(data)
RESULTS = ROOT / 'src/results'
OUT = ROOT / 'notebooks/outputs'
OUT.mkdir(parents=True, exist_ok=True)
print(f'{n} training timestamps; {len(future_times)} forecast timestamps; raw files: {DATA_DIR}')


## Recipe and validation

Fit group-level SARIMA `(1,0,1) × (2,1,0,12/24)`. Compute `p12 − line(p12) + line(p24)`, bound to [0,100], then merge by VM counts. Both models share fixed component orders; their period-specific orders were not optimized independently. Validate at origins 62 and 86; final origin 110 predicts 24 timestamps.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('submission_entry', ROOT / 'main.py')
entry = importlib.util.module_from_spec(spec)
spec.loader.exec_module(entry)
predictions, manifest = entry.main(['--data-dir', str(DATA_DIR), '--results-dir', str(RESULTS), '--evaluate'])
validate_predictions(predictions, future_times)
display(predictions)

## Main forecasts and all nine histories

Dotted observed holdout values are shown retrospectively after predictions have been written; they are not model inputs.

In [ ]:
display(Image(filename=str(RESULTS / 'figures/forecast.png')))
display(Image(filename=str(RESULTS / 'figures/groups_full_history.png')))

## Information availability

Forecasting uses training CPU readings and static segment metadata. Source lifetime CPU summaries, deletion timestamps and holdout-derived segment assignments are excluded. Relative timestamps are seconds on a 300-second grid. Public-window results remain retrospective because those observations were inspected during development.